# CS-317: Reinforcement Learning — Lab 10
## Multi-Step TD Learning and SARSA(λ)
**BS AI-1**

This notebook provides the long-aisle and charging-room environments and helper functions. Complete every `# TODO` cell, run all cells top to bottom without errors, and answer the written questions in the markdown cells provided for them.

- Set your personal **SEED** (the last three digits of your roll number) in the first code cell below. Every random experiment must use it, and your written answers must refer to your own numbers.
- **Predict first** cells must be filled in *before* you run the code beneath them.
- Your code must learn only through `env.reset()` and `env.step()`. `V_exact` is for **checking only**.

Reinforcement-learning libraries may not be used. NumPy and Matplotlib only.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.set_printoptions(precision=3, suppress=True)

In [ ]:
# ---------------------------------------------------------------
# Personal seed: the last three digits of your roll number.
# Every random experiment in this notebook must use it.
# ---------------------------------------------------------------
SEED = None          # TODO: e.g. SEED = 417

assert SEED is not None, "Set SEED to the last three digits of your roll number before running the notebook."
print("SEED =", SEED)

---
## Provided Starter Code

Run the cells in this section as-is. You do not need to modify them.

| Name | What it gives you |
|---|---|
| `AisleEnv()` | the long aisle: `reset()` → `s` ; `step(a)` → `s2, r, done` (rules hidden) |
| `random_policy` | the policy evaluated in Parts 1 and 2 (Left or Right, 0.5 each) |
| `run_episode(env, policy, rng)` | → `states, actions, rewards, next_states` (one entry per step) |
| `V_exact`, `rms_error(V)` | exact aisle values (checking only) and the RMS error over cells 1–9 |
| `plot_values(Vs, labels, title)` | bar plot of one or more estimates against the exact values |
| `A_states`, `A_rewards`, `A_next_states` | episode A (Tasks 1 and 4) |
| `RoomEnv()` | the charging room (Part 3) |
| `epsilon_greedy(Q, s, epsilon, rng)` | ε-greedy action choice, as in Lab 9 |
| `plot_room(Q, title)`, `cells_learned(Q)` | draws the room with the learned best action in every cell that has learned something, and counts those cells |

In [ ]:
# ---------------------------------------------------------------
# The long aisle (Parts 1 and 2)
#
#   cell:  0   1   2   3   4   5   6   7   8   9   10
#          F   .   .   .   .   S   .   .   .   .   G
#
#   Cells 0 and 10 end the episode. The robot starts in cell 5.
#   Actions: 0 = Left, 1 = Right (moves are not slippery).
#   Reward: +1 for a move that enters cell 10, 0 for every other move.
#   No discounting (gamma = 1).
# ---------------------------------------------------------------

AISLE_N = 11                       # cells 0..10
AISLE_START, AISLE_TERMINALS = 5, (0, 10)
AISLE_GAMMA = 1.0


class AisleEnv:
    """The long aisle as an environment you can only ACT in.

    reset()       -> s                 the start cell (5)
    step(action)  -> s2, r, done       next cell, reward, end-of-episode flag
    The rules are hidden: reading env.P or env.R raises an error.
    """

    def __init__(self):
        self.__s = None

    def __getattr__(self, name):
        if name in ("P", "R", "transition_matrix"):
            raise AttributeError(f"The model is hidden: '{name}' is not available. "
                                 "This environment only offers reset() and step(action).")
        raise AttributeError(name)

    def reset(self):
        self.__s = AISLE_START
        return self.__s

    def step(self, action):
        if self.__s is None or self.__s in AISLE_TERMINALS:
            raise RuntimeError("Episode has ended (or not started): call reset() first.")
        s2 = self.__s + (1 if action == 1 else -1)
        self.__s = s2
        return s2, (1.0 if s2 == 10 else 0.0), s2 in AISLE_TERMINALS


# The policy that is evaluated in Parts 1 and 2: Left or Right with probability 0.5 each
random_policy = np.full((AISLE_N, 2), 0.5)

# Exact values of the random policy (for CHECKING only): V(k) = k / 10, and 0 at the ends
V_exact = np.arange(AISLE_N) / 10.0
V_exact[list(AISLE_TERMINALS)] = 0.0


def run_episode(env, policy, rng, max_steps=10_000):
    """Plays one episode following `policy` and returns four lists, one entry per step:
    states [S0..S_{T-1}], actions [A0..A_{T-1}], rewards [R1..R_T], next_states [S1..S_T]."""
    s = env.reset()
    states, actions, rewards, next_states = [], [], [], []
    for _ in range(max_steps):
        a = int(rng.choice(policy.shape[1], p=policy[s]))
        s2, r, done = env.step(a)
        states.append(int(s)); actions.append(a); rewards.append(float(r)); next_states.append(int(s2))
        s = s2
        if done:
            break
    return states, actions, rewards, next_states


def rms_error(V):
    """Root-mean-square error of V over the non-terminal cells 1..9, against V_exact."""
    return float(np.sqrt(np.mean((np.asarray(V)[1:10] - V_exact[1:10]) ** 2)))


def plot_values(Vs, labels, title=""):
    """Bar plot of one or more value estimates over cells 1..9, with the exact values as a line."""
    cells = np.arange(1, 10)
    w = 0.8 / len(Vs)
    fig, ax = plt.subplots(figsize=(8, 3.4))
    for i, (V, lab) in enumerate(zip(Vs, labels)):
        ax.bar(cells - 0.4 + w / 2 + i * w, np.asarray(V)[1:10], w, label=lab)
    ax.plot(cells, V_exact[1:10], "k--", marker="o", label="exact")
    ax.set_xticks(cells); ax.set_xlabel("cell"); ax.set_ylabel("V(cell)")
    ax.set_title(title); ax.legend(fontsize=8); plt.tight_layout(); plt.show()


# Episode A (Tasks 1 and 4): a lucky game that walks straight to G
A_states      = [5, 6, 7, 8, 9]
A_rewards     = [0, 0, 0, 0, 1]
A_next_states = [6, 7, 8, 9, 10]

print("Exact values:", V_exact)

In [ ]:
# ---------------------------------------------------------------
# The charging room (Part 3)
#
#   A 5 x 10 room, cells numbered row by row: cell = 10 * row + col.
#   The robot starts at S (row 2, col 0 = cell 20) and must find the
#   charger C (row 2, col 9 = cell 29).
#   Actions: 0 = Up, 1 = Down, 2 = Left, 3 = Right. A move into a wall
#   leaves the robot in place. Reward: +1 for reaching C, 0 otherwise.
#   Reaching C ends the episode. Discount gamma = 0.9.
# ---------------------------------------------------------------

ROOM_ROWS, ROOM_COLS = 5, 10
ROOM_N, ROOM_ACTIONS = ROOM_ROWS * ROOM_COLS, 4
ROOM_START, ROOM_GOAL = 20, 29
ROOM_GAMMA = 0.9


class RoomEnv:
    """The charging room as an environment you can only ACT in.

    reset()       -> s                 the start cell (20)
    step(action)  -> s2, r, done       next cell, reward, end-of-episode flag
    """

    def __init__(self):
        self.__s = None

    def __getattr__(self, name):
        if name in ("P", "R", "transition_matrix"):
            raise AttributeError(f"The model is hidden: '{name}' is not available. "
                                 "This environment only offers reset() and step(action).")
        raise AttributeError(name)

    def reset(self):
        self.__s = ROOM_START
        return self.__s

    def step(self, action):
        if self.__s is None or self.__s == ROOM_GOAL:
            raise RuntimeError("Episode has ended (or not started): call reset() first.")
        r, c = divmod(self.__s, ROOM_COLS)
        if action == 0:   r = max(r - 1, 0)
        elif action == 1: r = min(r + 1, ROOM_ROWS - 1)
        elif action == 2: c = max(c - 1, 0)
        else:             c = min(c + 1, ROOM_COLS - 1)
        self.__s = r * ROOM_COLS + c
        return self.__s, (1.0 if self.__s == ROOM_GOAL else 0.0), self.__s == ROOM_GOAL


def epsilon_greedy(Q, s, epsilon, rng):
    """As in Lab 9: a random action with probability epsilon, otherwise a best action
    of Q[s] (ties broken at random)."""
    if rng.random() < epsilon:
        return int(rng.integers(Q.shape[1]))
    return int(rng.choice(np.flatnonzero(Q[s] == Q[s].max())))


def plot_room(Q, title=""):
    """Draws the room. In every cell where the robot has learned something (max Q > 0),
    an arrow shows its current best action; cells whose Q-values are still all zero are
    left blank. The title reports how many cells have learned something."""
    fig, ax = plt.subplots(figsize=(8, 4.2))
    learned = 0
    for s in range(ROOM_N):
        r, c = divmod(s, ROOM_COLS)
        face = "#c6e0b4" if s == ROOM_GOAL else ("#bdd7ee" if s == ROOM_START else "white")
        ax.add_patch(plt.Rectangle((c - .5, r - .5), 1, 1, facecolor=face, edgecolor="grey"))
        if s == ROOM_GOAL:
            ax.text(c, r, "C", ha="center", va="center", fontsize=13, fontweight="bold"); continue
        if s == ROOM_START:
            ax.text(c - .35, r - .3, "S", fontsize=10, fontweight="bold")
        if Q is not None and Q[s].max() > 0:
            learned += 1
            a = int(np.argmax(Q[s]))
            dx, dy = [(0, -.3), (0, .3), (-.3, 0), (.3, 0)][a]
            ax.annotate("", xy=(c + dx, r + dy), xytext=(c, r),
                        arrowprops=dict(arrowstyle="->", color="crimson", lw=1.5))
    ax.set_xlim(-.5, ROOM_COLS - .5); ax.set_ylim(ROOM_ROWS - .5, -.5)
    ax.set_xticks(range(ROOM_COLS)); ax.set_yticks(range(ROOM_ROWS))
    ax.set_xlabel("column"); ax.set_ylabel("row"); ax.set_aspect("equal")
    extra = "" if Q is None else f"  (learned something in {learned} cells)"
    ax.set_title(title + extra); plt.tight_layout(); plt.show()


def cells_learned(Q):
    """Number of cells in which at least one Q-value is above 0."""
    return int((Q.max(axis=1) > 0).sum())


plot_room(None, title="Charging room: S = start, C = charger")

---
## Part 1: Looking Further Ahead

One-step TD (Lab 8) builds its target from **one** real reward and then a guess. An **n-step return** uses **n** real rewards before it bootstraps:

$$G_t^{(n)} = R_{t+1} + \gamma R_{t+2} + \dots + \gamma^{n-1} R_{t+n} + \gamma^n V(S_{t+n})$$

If the episode ends before step $n$, stop at the end and leave out the guess. The update is unchanged; only the target is different:

$$V(S_t) \leftarrow V(S_t) + \alpha\,[\,G_t^{(n)} - V(S_t)\,]$$

The **λ-return** mixes all the n-step returns, giving weight $(1-\lambda)\lambda^{n-1}$ to the n-step return and the remaining weight to the full return. With $\lambda = 0$ it is the one-step target; with $\lambda = 1$ it is the full return.

### Task 1: Multi-Step Returns by Hand

Episode A walks straight from cell 5 to G: 5 → 6 → 7 → 8 → 9 → G, with rewards 0, 0, 0, 0, +1. Use $\gamma = 1$ and this table of current guesses:

| Cell | 5 | 6 | 7 | 8 | 9 |
|---|---|---|---|---|---|
| Current V | 0.3 | 0.2 | 0.4 | 0.6 | 0.8 |

**Predict first.** Which target do you expect to be closest to the true value V(5) = 0.5: the 1-step target or the full return? Why?

*your prediction here*

**(a)** Table 4: targets for cell 5 (the first step of episode A)

| Target | Calculation | Value |
|---|---|---|
| 1-step $G^{(1)}$ | | |
| 2-step $G^{(2)}$ | | |
| 3-step $G^{(3)}$ | | |
| 4-step $G^{(4)}$ | | |
| Full return $G$ | | |
| λ-return, λ = 0.5 | | |

**(b)** Update V(5) = 0.3 with α = 0.5 three times, once with each of these targets: the 1-step target, the full return, and the λ-return.

*your working here*

**(c)** Implement `n_step_return(rewards, next_states, t, n, V, gamma)` and use it to check every n-step row of Table 4.

In [ ]:
def n_step_return(rewards, next_states, t, n, V, gamma):
    # The n-step return from step t of a recorded episode:
    #   rewards[t] + gamma*rewards[t+1] + ... (n rewards, or fewer if the episode ends)
    #   + gamma**n * V[next_states[t+n-1]]   only if the episode has not ended by then.
    # TODO
    # your code here

    return G


V_snapshot = np.zeros(AISLE_N)
V_snapshot[5:10] = [0.3, 0.2, 0.4, 0.6, 0.8]

# TODO: print the 1-, 2-, 3-, 4-step and full return for t = 0 of episode A, and the
#       lambda-return with lambda = 0.5.
# your code here

**Questions**

1. Which target was closest to the true value V(5) = 0.5? Which was furthest? Does this mean longer targets are always better?

   *your answer here*

2. The 1-step target uses only V(6). Why is it so low here, even though the robot actually reached G?

   *your answer here*

### Task 2: n-Step TD

**(a)** Implement `n_step_td_episode(V, states, rewards, next_states, n, alpha, gamma)`. After a game has ended, it goes through the steps **in order** and moves each $V(S_t)$ towards its n-step return, using the current values.

In [ ]:
def n_step_td_episode(V, states, rewards, next_states, n, alpha, gamma):
    # For t = 0, 1, ..., T-1 (in order): V[states[t]] += alpha * (G_t^(n) - V[states[t]])
    # TODO (use n_step_return)
    # your code here

    pass


def n_step_td(env, policy, n, n_episodes, alpha, gamma, rng):
    # Plays n_episodes games and returns V and the RMS error after every game.
    V = np.zeros(AISLE_N)
    errors = np.zeros(n_episodes)
    # TODO: for every game: run_episode, then n_step_td_episode, then record rms_error(V)
    # your code here

    return V, errors

**Predict first.** For n = 1, 2, 4, 8 and 16, which n do you expect to give the smallest error after 10 games? After 50 games?

*your prediction here*

**(b)** For each n ∈ {1, 2, 4, 8, 16}, run 100 independent runs of 50 games (run $k$ uses `np.random.default_rng(SEED + k)`), with α = 0.3 and γ = 1. Plot the average RMS error against the number of games, one line per n, and complete Table 5.

In [ ]:
# TODO: 100 runs x 50 games for each n; average error curves on one labelled plot; Table 5.
# your code here

| n | Average error over games 1–10 | Error after game 10 | Error after game 50 |
|---|---|---|---|
| 1 | | | |
| 2 | | | |
| 4 | | | |
| 8 | | | |
| 16 | | | |

**Questions**

1. Which n learned fastest in the first 10 games, and which was most accurate after 50 games? Are they the same? Quote Table 5.

   *your answer here*

2. Explain the trade-off: what does a larger n gain, and what does it cost?

   *your answer here*

3. With n = 16, almost every target in the aisle is the full return. Which method from an earlier lab does n = 16 therefore behave like?

   *your answer here*

---
## Part 2: Eligibility Traces

Instead of waiting n steps, **TD(λ)** keeps a fading memory of the cells visited recently, called the **eligibility trace** $e(s)$. After every step, all traces fade and the current cell's trace goes up by 1; then the TD error updates **every** cell in proportion to its trace:

$$e(s) \leftarrow \gamma\lambda\, e(s) + \mathbb{1}[s = S_t] \qquad \delta_t = R_{t+1} + \gamma V(S_{t+1}) - V(S_t) \qquad V(s) \leftarrow V(s) + \alpha\,\delta_t\, e(s) \ \text{ for all } s$$

All traces start at 0 at the start of every episode.

### Task 3: TD(λ)

**(a)** By hand: episode B begins 5 → 6 → 5 → 6 → 7. Use γ = 1 and λ = 0.8, and complete Table 6 with the traces **after** each step's update.

| Step $t$ | Cell $S_t$ | $e(5)$ | $e(6)$ | $e(7)$ |
|---|---|---|---|---|
| 0 | 5 | | | |
| 1 | 6 | | | |
| 2 | 5 | | | |
| 3 | 6 | | | |
| 4 | 7 | | | |

If the TD error at step 4 is δ = 0.5 and α = 0.1, by how much does each of V(5), V(6) and V(7) change?

*your working here*

**(b)** Implement `td_lambda_episode(V, states, rewards, next_states, lam, alpha, gamma)` and `td_lambda(env, policy, lam, n_episodes, alpha, gamma, rng)`.

In [ ]:
def td_lambda_episode(V, states, rewards, next_states, lam, alpha, gamma):
    # Backward-view TD(lambda) with accumulating traces, applied to one recorded episode.
    E = np.zeros_like(V)                  # traces start at 0
    # TODO: for every step: compute delta, fade all traces, add 1 to the current cell's
    #       trace, then V += alpha * delta * E
    # your code here

    pass


def td_lambda(env, policy, lam, n_episodes, alpha, gamma, rng):
    # Plays n_episodes games and returns V and the RMS error after every game.
    V = np.zeros(AISLE_N)
    errors = np.zeros(n_episodes)
    # TODO
    # your code here

    return V, errors

**Predict first.** Which λ do you expect to learn fastest: 0, 0.4, 0.8 or 0.95?

*your prediction here*

**(c)** Repeat the experiment of Task 2 for λ ∈ {0, 0.4, 0.8, 0.95} (100 runs of 50 games, α = 0.3, γ = 1). Plot the average error curves and complete Table 7.

In [ ]:
# TODO: 100 runs x 50 games for each lambda; average error curves; Table 7.
# your code here

| λ | Average error over games 1–10 | Error after game 10 | Error after game 50 |
|---|---|---|---|
| 0 | | | |
| 0.4 | | | |
| 0.8 | | | |
| 0.95 | | | |

**Questions**

1. Compare your λ = 0 row with your n = 1 row of Table 5. What do you notice, and why?

   *your answer here*

2. Which λ learned fastest at the start? How does the λ that works best compare with the best n in Task 2?

   *your answer here*

3. In Table 6, the traces of cells 5 and 6 rose above 1. What caused that, and why does a cell that is visited often deserve more of the credit?

   *your answer here*

### Task 4: Watching Credit Spread

**Predict first.** After learning from episode A **once**, starting from V = 0, which cells will have changed under one-step TD? Under TD(λ = 0.8)?

*your prediction here*

Start each method from V = 0 and learn from episode A once, with α = 0.5 and γ = 1:

- one-step TD (n = 1),
- 4-step TD (n = 4),
- TD(λ) with λ = 0.8.

Plot the three value estimates with `plot_values` and complete Table 8.

In [ ]:
# TODO: learn from episode A once with each method (each from its own V = 0) and plot.
# your code here

| Method | V(5) | V(6) | V(7) | V(8) | V(9) |
|---|---|---|---|---|---|
| One-step TD | | | | | |
| 4-step TD | | | | | |
| TD(λ = 0.8) | | | | | |

**Questions**

1. In Lab 8 you found that with one-step TD, good news travels back only one step per game. How many cells did each method update after episode A?

   *your answer here*

2. Under TD(λ), why did the cells nearer the goal change more than the cells further away?

   *your answer here*

3. Why did 4-step TD leave V(5) unchanged, even though cell 5 was part of the successful game?

   *your answer here*

---
## Part 3: Faster Learning with SARSA(λ)

SARSA(λ) adds eligibility traces to the SARSA of Lab 9. Each **state–action pair** has a trace; after every step all traces fade by γλ, the pair just taken gets +1, and the SARSA TD error updates every pair in proportion to its trace:

$$\delta_t = R_{t+1} + \gamma Q(S_{t+1}, A_{t+1}) - Q(S_t, A_t) \qquad Q(s,a) \leftarrow Q(s,a) + \alpha\,\delta_t\, e(s,a) \ \text{ for all } (s, a)$$

With λ = 0 it is exactly the SARSA of Lab 9. In the charging room the only reward is +1 at the charger, so with plain SARSA the news of the charger spreads back very slowly.

### Task 5: SARSA(λ) in the Charging Room

**(a)** Implement `sarsa_lambda(env, n_episodes, alpha, epsilon, gamma, lam, rng)`. It returns the Q table (shape (50, 4), starting at 0) and the number of steps of every episode.

In [ ]:
def sarsa_lambda(env, n_episodes, alpha, epsilon, gamma, lam, rng):
    Q = np.zeros((ROOM_N, ROOM_ACTIONS))
    steps = np.zeros(n_episodes, dtype=int)
    for ep in range(n_episodes):
        E = np.zeros_like(Q)               # traces start at 0 in every episode
        s = env.reset()
        a = epsilon_greedy(Q, s, epsilon, rng)
        done = False
        while not done:
            s2, r, done = env.step(a)
            steps[ep] += 1
            # TODO: choose a2 with epsilon_greedy; compute delta (target = r if done);
            #       fade all traces by gamma*lam; add 1 to E[s, a];
            #       Q += alpha * delta * E; then s, a = s2, a2
            # your code here

    return Q, steps

**Predict first.** How many episodes do you think plain SARSA (λ = 0) and SARSA(λ = 0.9) will each need before they reach the charger in under 30 steps?

*your prediction here*

**(b)** For λ ∈ {0, 0.5, 0.9}, run 20 independent runs of 30 episodes (run $k$ uses seed SEED + $k$), with α = 0.1, ε = 0.1 and γ = 0.9. Plot the average number of steps per episode, one line per λ, and complete Table 9.

In [ ]:
# TODO: 20 runs x 30 episodes for each lambda; average steps per episode; Table 9.
# your code here

| λ | Steps in episode 1 | Average steps, episodes 2–5 | Episodes 6–10 | Episodes 21–30 | Total steps, 30 episodes |
|---|---|---|---|---|---|
| 0 (SARSA) | | | | | |
| 0.5 | | | | | |
| 0.9 | | | | | |

**(c)** With seed SEED, run plain SARSA (λ = 0) and SARSA(λ = 0.9) for **2 episodes** only. Draw both Q tables with `plot_room`, and record `cells_learned` for each.

Cells learned after 2 episodes: λ = 0: ___ , λ = 0.9: ___

In [ ]:
# TODO: two plot_room figures after 2 episodes (lambda = 0 and lambda = 0.9), and
#       cells_learned for each.
# your code here

**Questions**

1. Why did the first episode take the same number of steps for every λ? (Hint: what is in the Q table until the charger is reached?)

   *your answer here*

2. Use your two `plot_room` figures and the cells-learned counts to explain why SARSA(λ) found a good route so much sooner. Relate it to what you saw in Task 4.

   *your answer here*

3. Look at your episodes 21–30 column. Did the advantage of traces last? Why might the methods end up similar after many episodes?

   *your answer here*

---
## Part 4: Summary and Reflection

### Task 6: Summary and Reflection

**(a)** Table 10: How long is the return?

| Method | How many real rewards before the guess? | Prediction or control? | Lab |
|---|---|---|---|
| TD(0) | | | |
| n-step TD | | | |
| TD(λ) | | | |
| Monte Carlo | | | |
| SARSA | | | |
| SARSA(λ) | | | |

**(b)** Explain how n-step TD and eligibility traces help credit reach earlier states, and name one cost of looking further ahead. Quote at least two numbers from your own results.